# Cluster Profile And Interpretation

This notebook reads the final clustered dataset produced by teste.ipynb and turns the numeric clusters into an academic interpretation. It profiles cluster size, funding magnitude, funding composition, status distribution, and market concentration.

## Interpretation Rules

- Cluster IDs are arbitrary numeric labels from K-Means.
- Names are assigned after profiling the saved labels.
- Median values are emphasized because funding amounts are strongly skewed.
- Statistical tests are descriptive support, not causal evidence.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, kruskal

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

PROFILE_DATA = Path("clustered_startups_real_values.csv")
df_profile = pd.read_csv(PROFILE_DATA)

assert "Cluster" in df_profile.columns, "Run teste.ipynb first to generate the Cluster column."
print(f"Loaded {len(df_profile):,} clustered startups from {PROFILE_DATA}")
display(df_profile.head())

## Output Integrity Checks

These checks confirm the profile notebook is using the same saved cluster labels produced by the modeling notebook.

In [ ]:
required_columns = [
    "name",
    "market",
    "funding_total_usd",
    "funding_rounds",
    "early_stage_funding",
    "venture",
    "debt_financing",
    "private_equity",
    "stage_level",
    "Cluster",
]
missing_required = [col for col in required_columns if col not in df_profile.columns]
assert not missing_required, f"Missing required columns: {missing_required}"

cluster_sizes = (
    df_profile["Cluster"]
    .value_counts()
    .sort_index()
    .rename("Count")
    .to_frame()
)
cluster_sizes["Percentage"] = cluster_sizes["Count"] / cluster_sizes["Count"].sum() * 100

assert (cluster_sizes["Count"] > 0).all()
display(cluster_sizes)

## Real-Value Cluster Summary

The median profile is the main table for interpretation. Means are also shown because they reveal the impact of very large funding outliers.

In [ ]:
real_features = [
    "funding_total_usd",
    "funding_rounds",
    "early_stage_funding",
    "venture",
    "debt_financing",
    "private_equity",
    "stage_level",
]
real_features = [col for col in real_features if col in df_profile.columns]

cluster_median_profile = df_profile.groupby("Cluster")[real_features].median().round(0)
cluster_mean_profile = df_profile.groupby("Cluster")[real_features].mean().round(0)

display(cluster_median_profile)
display(cluster_mean_profile)

In [ ]:
def format_usd(value):
    if pd.isna(value):
        return ""
    if abs(value) >= 1_000_000_000:
        return f"${value / 1_000_000_000:.2f}B"
    if abs(value) >= 1_000_000:
        return f"${value / 1_000_000:.2f}M"
    if abs(value) >= 1_000:
        return f"${value / 1_000:.1f}K"
    return f"${value:,.0f}"

money_cols = [
    "funding_total_usd",
    "early_stage_funding",
    "venture",
    "debt_financing",
    "private_equity",
]

cluster_summary = cluster_sizes.join(cluster_median_profile)
format_dict = {col: format_usd for col in money_cols if col in cluster_summary.columns}
format_dict.update({"Percentage": "{:.2f}%", "funding_rounds": "{:.0f}", "stage_level": "{:.0f}"})

display(
    cluster_summary
    .style
    .format(format_dict)
    .set_caption("Cluster Summary With Real Median Values")
)

## Funding Composition

Funding shares show whether a cluster is defined by early-stage funding, venture, debt, or private equity rather than only by total funding size.

In [ ]:
share_sources = {
    "early_stage_share": "early_stage_funding",
    "venture_share": "venture",
    "debt_share": "debt_financing",
    "private_equity_share": "private_equity",
}

for share_col, value_col in share_sources.items():
    df_profile[share_col] = np.where(
        df_profile["funding_total_usd"] > 0,
        df_profile[value_col] / df_profile["funding_total_usd"],
        0,
    ).clip(0, 1)

share_cols = list(share_sources.keys())
cluster_funding_share = df_profile.groupby("Cluster")[share_cols].median().round(3)

display(
    cluster_funding_share
    .style
    .format("{:.1%}")
    .set_caption("Median Funding Composition By Cluster")
)

## Status And Market Profiles

Status distribution and market concentration help describe whether clusters differ in outcomes or sector mix.

In [ ]:
if "status" in df_profile.columns:
    status_by_cluster = pd.crosstab(df_profile["Cluster"], df_profile["status"], normalize="index") * 100
    status_by_cluster = status_by_cluster.round(2)
    display(status_by_cluster.style.format("{:.2f}%").set_caption("Company Status Distribution By Cluster"))

    raw_status_table = pd.crosstab(df_profile["Cluster"], df_profile["status"])
    chi2, p_value, dof, expected = chi2_contingency(raw_status_table)
    display(pd.DataFrame([{"test": "chi-square status vs cluster", "chi2": chi2, "p_value": p_value, "dof": dof}]))

top_markets_by_cluster = (
    df_profile
    .groupby("Cluster")["market"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("Percentage")
    .reset_index()
    .sort_values(["Cluster", "Percentage"], ascending=[True, False])
)

display(top_markets_by_cluster.groupby("Cluster").head(10))

## Numeric Differences Across Clusters

Kruskal-Wallis tests are used because funding variables are skewed and non-normal. These tests support whether distributions differ across clusters, but they do not explain causality.

In [ ]:
kruskal_rows = []

for col in real_features:
    groups = [
        df_profile.loc[df_profile["Cluster"] == cluster, col].dropna()
        for cluster in sorted(df_profile["Cluster"].unique())
    ]
    if len(groups) >= 2 and all(len(group) > 0 for group in groups):
        statistic, p_value = kruskal(*groups)
        kruskal_rows.append({"feature": col, "kruskal_statistic": statistic, "p_value": p_value})

kruskal_results = pd.DataFrame(kruskal_rows).sort_values("p_value")
display(kruskal_results)

## Visual Profiles

The bar charts compare cluster medians. Boxplots hide extreme outliers so the central distribution remains readable.

In [ ]:
for col in real_features:
    medians = df_profile.groupby("Cluster")[col].median().sort_index()

    plt.figure(figsize=(8, 5))
    plt.bar(medians.index.astype(str), medians.values)
    plt.title(f"Median {col} by cluster")
    plt.xlabel("Cluster")
    plt.ylabel(col)

    if col in money_cols and (medians > 0).any():
        plt.yscale("log")
        plt.ylabel(f"{col} - log scale")

    plt.show()

In [ ]:
for col in real_features:
    data = [
        df_profile.loc[df_profile["Cluster"] == cluster, col].dropna()
        for cluster in sorted(df_profile["Cluster"].unique())
    ]

    plt.figure(figsize=(8, 5))
    plt.boxplot(data, labels=sorted(df_profile["Cluster"].unique()), showfliers=False)
    plt.title(f"Distribution of {col} by cluster")
    plt.xlabel("Cluster")
    plt.ylabel(col)

    if col in money_cols:
        positive_groups = [group[group > 0] for group in data if (group > 0).any()]
        if positive_groups:
            plt.yscale("log")
            plt.ylabel(f"{col} - log scale")

    plt.show()

## Assign Interpretable Cluster Names

Names are assigned from the profile itself, not assumed before modeling. This matters because numeric K-Means labels are arbitrary.

In [ ]:
profile_for_names = cluster_median_profile.join(cluster_funding_share, how="left")
cluster_names = {}

available = set(profile_for_names.index)

if "private_equity_share" in profile_for_names.columns and available:
    private_equity_cluster = profile_for_names.loc[list(available), "private_equity_share"].idxmax()
    cluster_names[private_equity_cluster] = "Private equity / capital intensive"
    available.remove(private_equity_cluster)

if "debt_share" in profile_for_names.columns and available:
    debt_cluster = profile_for_names.loc[list(available), "debt_share"].idxmax()
    cluster_names[debt_cluster] = "Debt-financed startups"
    available.remove(debt_cluster)

if "venture_share" in profile_for_names.columns and available:
    venture_cluster = profile_for_names.loc[list(available), "venture_share"].idxmax()
    cluster_names[venture_cluster] = "Venture-backed growth"
    available.remove(venture_cluster)

for cluster in sorted(available):
    cluster_names[cluster] = "Early-stage / low funding"

df_profile["Cluster_Name"] = df_profile["Cluster"].map(cluster_names)
cluster_name_table = (
    df_profile[["Cluster", "Cluster_Name"]]
    .drop_duplicates()
    .sort_values("Cluster")
    .reset_index(drop=True)
)
display(cluster_name_table)

## Final Cluster Report

This table combines size, median funding profile, funding composition, and status distribution into a single report suitable for the final academic write-up.

In [ ]:
final_cluster_report = cluster_summary.join(cluster_funding_share)

if "status" in df_profile.columns:
    status_percentages = pd.crosstab(df_profile["Cluster"], df_profile["status"], normalize="index") * 100
    final_cluster_report = final_cluster_report.join(status_percentages.round(2))

final_cluster_report.insert(0, "Cluster_Name", final_cluster_report.index.map(cluster_names))

display(final_cluster_report)

## Interpretation Notes

- Early-stage / low funding: lower median funding, fewer rounds, and smaller funding subtype amounts.
- Debt-financed startups: comparatively high debt share and debt-financing median.
- Private equity / capital intensive: high total funding and private-equity concentration.
- Venture-backed growth: high venture funding, more funding rounds, and stronger stage progression.

These names describe observed funding patterns. They should not be treated as fixed startup types outside this dataset.

## Limitations

The dataset is sparse and may omit private or unreported funding events. Some funding subtype totals exceed total funding, so ratios are clipped after being reported in the modeling notebook. The model is sensitive to feature choices, scaling, and the selected k. The clusters are descriptive segments for exploratory analysis, not causal explanations or predictions of startup success.